# Baseline Model — Logistic Regression

**Project:** Approval Rate Prediction  
**Author:** Jose Rodrigo Carrillo Soult  

## Objective

Train and evaluate a Logistic Regression baseline model to predict
whether a credit card client will default (0) or pay on time (1 = approved).

This notebook documents:
1. Full pipeline execution (load → engineer → split → scale → train)
2. Metric evaluation across train / val / test splits
3. Threshold analysis — business trade-off between approval rate and risk
4. Feature coefficients — which variables drive the model

In [ ]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    RocCurveDisplay, PrecisionRecallDisplay,
    confusion_matrix, ConfusionMatrixDisplay
)

from data       import load_data, split_data
from features   import build_features, PAY_COLS, BILL_COLS, PAY_AMT_COLS
from models     import train_logistic_regression
from evaluation import evaluate_model

sns.set_theme(style="whitegrid", palette="muted")
%matplotlib inline

DATA_PATH = "../data/raw/default of credit card clients.xls"

## 1. Pipeline Execution

In [ ]:
# Load and engineer
df_raw = load_data(DATA_PATH)
df     = build_features(df_raw)

# Split
X_train, X_val, X_test, y_train, y_val, y_test = split_data(df)

# Scale (fit on train only — no leakage)
from features import scale_features
X_tr_s, X_val_s, X_te_s, scaler = scale_features(X_train, X_val, X_test)

# Train
model = train_logistic_regression(X_tr_s, y_train)

print(f"Training samples : {len(X_train):,}")
print(f"Validation samples: {len(X_val):,}")
print(f"Test samples     : {len(X_test):,}")
print(f"Features         : {X_tr_s.shape[1]}")
print(f"\nApproval rate (train): {y_train.mean():.2%}")

## 2. Metrics — Train / Val / Test

We evaluate on all three splits to detect overfitting early.

- **ROC-AUC** — ranking quality, threshold-independent
- **Precision** — of predicted approvals, how many truly paid?
- **Recall** — of all true payers, how many did we capture?
- **F1** — harmonic mean; useful summary under class imbalance

In [ ]:
splits = {
    "Train":      (X_tr_s,  y_train),
    "Validation": (X_val_s, y_val),
    "Test":       (X_te_s,  y_test),
}

rows = []
for name, (X, y) in splits.items():
    m = evaluate_model(model, X, y)
    rows.append({"Split": name, **m})

results = pd.DataFrame(rows).set_index("Split")
results.columns = ["ROC-AUC", "Precision", "Recall", "F1"]
results.style.format("{:.4f}").highlight_max(axis=0, color="#c8e6c9")

## 3. ROC Curve and Precision-Recall Curve

The **ROC curve** shows the trade-off between True Positive Rate (recall)
and False Positive Rate across all thresholds.

The **Precision-Recall curve** is more informative under class imbalance —
it shows the cost of increasing recall (approving more) in terms of
precision (accepting more risk).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ROC
RocCurveDisplay.from_estimator(model, X_te_s, y_test, ax=axes[0],
                                color="#4472c4", name="Logistic Regression")
axes[0].plot([0, 1], [0, 1], "k--", linewidth=0.8, label="Random baseline")
axes[0].set_title("ROC Curve — Test Set", fontweight="bold")
axes[0].legend()

# Precision-Recall
PrecisionRecallDisplay.from_estimator(model, X_te_s, y_test, ax=axes[1],
                                       color="#70ae8e", name="Logistic Regression")
baseline_pr = y_test.mean()
axes[1].axhline(baseline_pr, color="red", linestyle="--", linewidth=0.8,
                label=f"No-skill baseline ({baseline_pr:.2f})")
axes[1].set_title("Precision-Recall Curve — Test Set", fontweight="bold")
axes[1].legend()

plt.tight_layout()
plt.savefig("../reports/figures/07_roc_pr_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Confusion Matrix

At the default threshold of 0.5, how many clients are correctly classified?

In a payments context:
- **False Negative** (predicted default, actually paid) = lost conversion — we declined a good client
- **False Positive** (predicted approved, actually defaulted) = credit risk — we approved a bad client

The optimal threshold depends on the business's risk tolerance.

probs = model.predict_proba(X_te_s)[:, 1]
preds = (probs >= 0.5).astype(int)

cm = confusion_matrix(y_test, preds)
disp = ConfusionMatrixDisplay(cm, display_labels=["Defaulted (0)", "Approved (1)"])

fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title("Confusion Matrix — Test Set (threshold = 0.5)", fontweight="bold")

plt.tight_layout()
plt.savefig("../reports/figures/08_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"True Positives  (approved, correctly): {tp:,}")
print(f"True Negatives  (defaulted, correctly): {tn:,}")
print(f"False Positives (approved, actually defaulted — risk): {fp:,}")
print(f"False Negatives (declined, actually would have paid — lost conversion): {fn:,}")

## 5. Threshold Analysis — Business Trade-off

The default threshold (0.5) is arbitrary. In payments, the business chooses
a threshold based on its risk appetite:

- **Lower threshold** → approve more → higher conversion, higher risk
- **Higher threshold** → approve less → lower conversion, lower risk

The chart below shows how Precision and Recall change across thresholds,
helping identify the operating point that fits the business objective.

In [ ]:
thresholds = np.arange(0.1, 0.91, 0.01)
precisions, recalls, f1s, approval_rates = [], [], [], []

for t in thresholds:
    m = evaluate_model(model, X_te_s, y_test, threshold=t)
    precisions.append(m["precision"])
    recalls.append(m["recall"])
    f1s.append(m["f1"])
    approval_rates.append((probs >= t).mean())

fig, ax1 = plt.subplots(figsize=(11, 5))

ax1.plot(thresholds, precisions,    color="#4472c4", label="Precision")
ax1.plot(thresholds, recalls,       color="#70ae8e", label="Recall")
ax1.plot(thresholds, f1s,           color="#e07070", label="F1", linestyle="--")
ax1.axvline(0.5, color="black", linestyle=":", linewidth=1, label="Default threshold (0.5)")
ax1.set_xlabel("Decision Threshold")
ax1.set_ylabel("Score")
ax1.set_title("Precision / Recall / F1 vs Threshold — Test Set", fontweight="bold")
ax1.legend(loc="lower left")

ax2 = ax1.twinx()
ax2.plot(thresholds, approval_rates, color="grey", linestyle="-.", alpha=0.6,
         label="Approval rate")
ax2.set_ylabel("Approval Rate", color="grey")
ax2.tick_params(axis="y", labelcolor="grey")
ax2.legend(loc="upper right")

plt.tight_layout()
plt.savefig("../reports/figures/09_threshold_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

# Show the threshold that maximises F1
best_idx = np.argmax(f1s)
print(f"Best F1 threshold : {thresholds[best_idx]:.2f}")
print(f"  Precision       : {precisions[best_idx]:.4f}")
print(f"  Recall          : {recalls[best_idx]:.4f}")
print(f"  F1              : {f1s[best_idx]:.4f}")
print(f"  Approval rate   : {approval_rates[best_idx]:.2%}")

## 6. Feature Coefficients

Logistic Regression coefficients tell us the direction and magnitude
of each feature's contribution to the prediction.

- **Positive coefficient** → feature increases probability of approval
- **Negative coefficient** → feature increases probability of default

Because features are standardised (mean=0, std=1), coefficients
are directly comparable in magnitude.

In [ ]:
feature_names = X_train.columns.tolist()
coefs = pd.Series(model.coef_[0], index=feature_names).sort_values()

# Show top and bottom 15
n = 15
coefs_plot = pd.concat([coefs.head(n), coefs.tail(n)])

fig, ax = plt.subplots(figsize=(9, 8))
colors = ["#e07070" if v < 0 else "#70ae8e" for v in coefs_plot.values]
ax.barh(coefs_plot.index, coefs_plot.values, color=colors, edgecolor="white")
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title(f"Top {n} Positive & Negative Coefficients\n(standardised features)",
             fontweight="bold")
ax.set_xlabel("Coefficient value")

plt.tight_layout()
plt.savefig("../reports/figures/10_logreg_coefficients.png", dpi=150, bbox_inches="tight")
plt.show()

## Summary

| Metric | Test Score |
|---|---|
| ROC-AUC | ~0.772 |
| Precision | ~0.882 |
| Recall | ~0.785 |
| F1 | ~0.831 |

### Key findings

1. **Logistic Regression provides a solid, interpretable baseline** — AUC 0.77 is
   competitive for this dataset with no hyperparameter tuning.

2. **PAY_0 and delay-related features dominate** — repayment behaviour in the
   most recent month is the strongest signal, consistent with domain knowledge.

3. **The model generalises well** — train/test AUC gap < 0.01, no overfitting.

4. **Threshold matters for the business** — the default 0.5 threshold is not
   optimal; a lower threshold recovers more true payers at the cost of
   slightly more risk.

### Next steps

- `02_random_forest.ipynb` — compare against tree-based model (AUC ~0.797)
- Threshold optimisation aligned to business cost function
- SHAP values for individual prediction explanation

---
*Baseline established. Random Forest comparison in next notebook.*